# Experiment 1 — Capacity Competition for Shared vs. Language-Specific Computation

### Purpose

This notebook tests the **theory itself before training any neural model**.

We compare allocations of a fixed capacity budget \(N\) between one shared circuit serving multiple languages and language-specific private circuits.

The central question is:

> Does the capacity-allocation model itself produce a transition from shared computation at small \(N\) to language-specific computation at large \(N\)?

We then test how that transition changes when we manipulate the theory's two key quantities:

\[
\tau_L = \text{shortcut richness},\qquad
\varepsilon_L = \text{shared-space distortion}.
\]

This is deliberately a **theory-validation experiment**, not a Transformer experiment.

### Model

For language \(L\), native/private computation has loss

\[
(1-\tau_L)c_a q(n_L),
\]

while shared computation has loss

\[
c_a q(n_S)+\varepsilon_L.
\]

We use the power-law special case

\[
q(n)=\alpha n^{-\beta}
\]

after formulating the constrained allocation.

The total capacity is

\[
n_S+\sum_{L\in P}n_L\le N,
\]

where \(P\) is the set of languages assigned private circuits.

The main output is the optimal private set

\[
P^*(N)=\arg\min_P \mathcal L_P^*(N).
\]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ALPHA = 1.0
C_A = 1.0
BETA = 0.5

np.set_printoptions(precision=4, suppress=True)
pd.set_option('display.max_columns', 20)


## 1. Exact allocation for a fixed private set

For a fixed private set \(P\), let \(K-|P|\) languages use the shared circuit.

Under

\[
q(n)=\alpha n^{-\beta},
\]

the constrained problem has a closed-form solution.

Define

\[
A_P=(K-|P|)c_a,\qquad B_L=(1-\tau_L)c_a.
\]

The optimal allocation satisfies

\[
n_L=n_S\left(\frac{B_L}{A_P}\right)^{1/(\beta+1)}
\]

and

\[
n_S=\frac{N}{1+\sum_{L\in P}(B_L/A_P)^{1/(\beta+1)}}.
\]

Substitution gives

\[
\boxed{\mathcal L_P^*(N)=
\frac{\alpha}{K}\left[A_P^{1/(\beta+1)}+\sum_{L\in P}B_L^{1/(\beta+1)}\right]^{\beta+1}N^{-\beta}
+\frac{1}{K}\sum_{L\notin P}\varepsilon_L.}
\]

In [ ]:
def optimal_allocation_for_subset(N, tau, eps, private_set,
                                  alpha=1.0, c_a=1.0, beta=0.5):
    tau = np.asarray(tau, dtype=float)
    eps = np.asarray(eps, dtype=float)
    K = len(tau)
    P = tuple(sorted(private_set))
    shared = tuple(i for i in range(K) if i not in P)
    m = len(shared)

    if m == 0:
        b = np.array([(1.0 - tau[i]) * c_a for i in P])
        weights = b ** (1.0 / (beta + 1.0))
        private_caps = {i: N * weights[j] / weights.sum() for j, i in enumerate(P)}
        approximation = alpha * (weights.sum() ** (beta + 1.0)) * N ** (-beta)
        return approximation / K, 0.0, private_caps

    if len(P) == 0:
        shared_capacity = float(N)
        loss = alpha * c_a * N ** (-beta) + eps.mean()
        return loss, shared_capacity, {}

    A = m * c_a
    ratios = np.array([
        (((1.0 - tau[i]) * c_a) / A) ** (1.0 / (beta + 1.0))
        for i in P
    ])
    shared_capacity = N / (1.0 + ratios.sum())
    private_caps = {i: shared_capacity * ratios[j] for j, i in enumerate(P)}

    approximation = A * alpha * shared_capacity ** (-beta)
    for i in P:
        approximation += alpha * (1.0 - tau[i]) * c_a * private_caps[i] ** (-beta)

    floor = sum(eps[i] for i in shared)
    loss = (approximation + floor) / K
    return loss, shared_capacity, private_caps


def closed_form_subset_loss(N, tau, eps, private_set, alpha=1.0, c_a=1.0, beta=0.5):
    tau = np.asarray(tau, dtype=float)
    eps = np.asarray(eps, dtype=float)
    K = len(tau)
    P = tuple(sorted(private_set))
    m = K - len(P)

    terms = []
    if m > 0:
        terms.append((m * c_a) ** (1.0 / (beta + 1.0)))
    for i in P:
        terms.append(((1.0 - tau[i]) * c_a) ** (1.0 / (beta + 1.0)))

    approx_coeff = alpha / K * sum(terms) ** (beta + 1.0)
    floor = sum(eps[i] for i in range(K) if i not in P) / K
    return approx_coeff * N ** (-beta) + floor


def best_private_subset(N, tau, eps, alpha=1.0, c_a=1.0, beta=0.5):
    K = len(tau)
    best_loss = np.inf
    best_set = None
    best_shared = None
    best_private_caps = None

    for bits in range(1 << K):
        P = tuple(i for i in range(K) if (bits >> i) & 1)
        loss, shared_cap, private_caps = optimal_allocation_for_subset(
            N, tau, eps, P, alpha=alpha, c_a=c_a, beta=beta
        )
        if loss < best_loss:
            best_loss = loss
            best_set = P
            best_shared = shared_cap
            best_private_caps = private_caps

    return {'loss': best_loss, 'private_set': best_set, 'num_private': len(best_set),
            'shared_capacity': best_shared, 'private_capacities': best_private_caps}


## 2. Sanity check: exact allocation vs. closed form

In [ ]:
rng = np.random.default_rng(0)
K_test = 5
tau_test = rng.uniform(0.0, 0.5, K_test)
eps_test = rng.uniform(0.0, 0.1, K_test)
P_test = (1, 3)
N_test = 40.0

direct = optimal_allocation_for_subset(N_test, tau_test, eps_test, P_test, alpha=ALPHA, c_a=C_A, beta=BETA)[0]
closed = closed_form_subset_loss(N_test, tau_test, eps_test, P_test, alpha=ALPHA, c_a=C_A, beta=BETA)
assert abs(direct - closed) < 1e-10
print('Sanity check passed.')
print('Direct loss :', direct)
print('Closed-form :', closed)


# 3. Symmetric case

First impose

\[
\tau_L=\tau,\qquad \varepsilon_L=\varepsilon.
\]

Then every subset with \(r\) private languages has the same loss, so the optimization reduces to \(r^*(N)\in\{0,1,\ldots,K\}\).

This is an important diagnostic: **does symmetry by itself produce a smooth language-by-language cascade, or does it mainly produce a global crossover?**

In [ ]:
def homogeneous_subset_loss(N, K, r, tau, eps, alpha=1.0, c_a=1.0, beta=0.5):
    tau_arr = np.full(K, tau, dtype=float)
    eps_arr = np.full(K, eps, dtype=float)
    return closed_form_subset_loss(N, tau_arr, eps_arr, tuple(range(r)), alpha=alpha, c_a=c_a, beta=beta)


def homogeneous_best_r(N, K, tau, eps, alpha=1.0, c_a=1.0, beta=0.5):
    losses = np.array([homogeneous_subset_loss(N, K, r, tau, eps, alpha, c_a, beta) for r in range(K+1)])
    return int(np.argmin(losses)), losses

K = 8
tau = 0.20
eps = 0.10
Ns = np.logspace(-1, 5, 500)
r_star = np.array([homogeneous_best_r(N, K, tau, eps, ALPHA, C_A, BETA)[0] for N in Ns])
print('Distinct optimal r values:', sorted(np.unique(r_star).tolist()))


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(Ns, r_star, linewidth=2)
plt.xscale('log')
plt.xlabel('Total capacity N')
plt.ylabel('Optimal number of private languages r*(N)')
plt.title('Symmetric case')
plt.grid(True, alpha=0.25)
plt.show()


### Interpretation

For many parameter regimes the symmetric model jumps directly from mostly/all shared to mostly/all private.

That is useful: it tells us that a **smooth language-by-language dissociation curve should not be assumed to come from symmetry**.

The natural next ingredient is heterogeneity in

\[
(\tau_L,\varepsilon_L),
\]

which gives languages different specialization thresholds.

# 4. Heterogeneous languages: specialization cascade

Now each language has its own shortcut richness and shared-space distortion.

We compute

\[
P^*(N)=\arg\min_P \mathcal L_P^*(N)
\]

by exhaustive enumeration. For \(K=8\), there are only \(2^8=256\) possible private sets, so exact enumeration is cheap.

In [ ]:
K = 8
tau_hetero = np.array([0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40])
eps_hetero = np.array([0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08])
Ns = np.logspace(-1, 5, 500)

records = []
best_sets = []
for N in Ns:
    result = best_private_subset(N, tau_hetero, eps_hetero, ALPHA, C_A, BETA)
    records.append({'N': N, 'loss': result['loss'], 'num_private': result['num_private'],
                    'private_set': result['private_set'], 'shared_capacity': result['shared_capacity']})
    best_sets.append(result['private_set'])
alloc_df = pd.DataFrame(records)
print('Distinct optimal numbers of private languages:', sorted(alloc_df['num_private'].unique().tolist()))


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.step(alloc_df['N'], alloc_df['num_private'], where='post', linewidth=2)
plt.xscale('log')
plt.xlabel('Total capacity N')
plt.ylabel('Optimal number of private languages')
plt.title('Heterogeneous case: specialization cascade')
plt.grid(True, alpha=0.25)
plt.show()


In [ ]:
first_seen = {}
for N, P in zip(Ns, best_sets):
    for lang in P:
        first_seen.setdefault(lang, N)
order_df = pd.DataFrame({
    'language': np.arange(K),
    'tau': tau_hetero,
    'epsilon': eps_hetero,
    'first_specialized_capacity': [first_seen.get(i, np.inf) for i in range(K)]
}).sort_values('first_specialized_capacity')
order_df


The desired qualitative phenomenon is now explicit:

\[
\text{shared}\rightarrow\text{partial specialization}\rightarrow\text{more specialization}.
\]

The smoothness comes from a **distribution of language-specific thresholds**, not from assuming that one language has a smooth binary transition.

# 5. Comparative statics: \(\varepsilon\)

Prediction:

\[
\boxed{\varepsilon\uparrow\Longrightarrow N^*\downarrow}
\]

We vary the shared-space distortion of one target language while keeping its shortcut richness fixed.

In [ ]:
def first_specialization_threshold(K, tau, eps, beta=0.5, alpha=1.0, c_a=1.0, N_grid=None):
    if N_grid is None:
        N_grid = np.logspace(-4, 8, 1800)
    for N in N_grid:
        result = best_private_subset(N, np.asarray(tau), np.asarray(eps), alpha, c_a, beta)
        if result['num_private'] > 0:
            return N
    return np.inf

K = 4
base_tau = np.full(K, 0.20)
eps_values = np.logspace(-3, 0, 12)
thresholds_eps = []
for e in eps_values:
    eps_vec = np.full(K, 0.02)
    eps_vec[-1] = e
    thresholds_eps.append(first_specialization_threshold(K, base_tau, eps_vec, BETA, ALPHA, C_A))
eps_df = pd.DataFrame({'epsilon_target': eps_values, 'first_specialization_capacity': thresholds_eps})
eps_df


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(eps_df['epsilon_target'], eps_df['first_specialization_capacity'], marker='o')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Shared-space distortion ε')
plt.ylabel('First specialization capacity')
plt.title('Prediction: larger ε causes earlier specialization')
plt.grid(True, alpha=0.25)
plt.show()


# 6. Comparative statics: \(\tau\)

Prediction:

\[
\boxed{\tau\uparrow\Longrightarrow N^*\downarrow}
\]

We vary the shortcut richness of one target language while holding its shared-space distortion fixed.

In [ ]:
tau_values = np.linspace(0.0, 0.8, 12)
thresholds_tau = []
for t in tau_values:
    tau_vec = np.full(K, 0.20)
    tau_vec[-1] = t
    eps_vec = np.full(K, 0.05)
    thresholds_tau.append(first_specialization_threshold(K, tau_vec, eps_vec, BETA, ALPHA, C_A))
tau_df = pd.DataFrame({'tau_target': tau_values, 'first_specialization_capacity': thresholds_tau})
tau_df


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(tau_df['tau_target'], tau_df['first_specialization_capacity'], marker='o')
plt.yscale('log')
plt.xlabel('Shortcut richness τ')
plt.ylabel('First specialization capacity')
plt.title('Prediction: larger τ causes earlier specialization')
plt.grid(True, alpha=0.25)
plt.show()


# 7. Joint phase diagram

The central theory prediction is a map

\[
(\tau,\varepsilon)\mapsto N_{\mathrm{first}}.
\]

We compute that surface directly from the allocation problem.

In [ ]:
K = 4
tau_grid = np.linspace(0.0, 0.75, 25)
eps_grid = np.logspace(-3, 0, 25)
N_grid = np.logspace(-4, 8, 1600)
phase = np.empty((len(eps_grid), len(tau_grid)))

for i, e in enumerate(eps_grid):
    for j, t in enumerate(tau_grid):
        tau_vec = np.full(K, 0.20)
        eps_vec = np.full(K, 0.05)
        tau_vec[-1] = t
        eps_vec[-1] = e
        phase[i, j] = first_specialization_threshold(K, tau_vec, eps_vec, BETA, ALPHA, C_A, N_grid)

phase_df = pd.DataFrame(phase, index=eps_grid, columns=tau_grid)
phase_df.iloc[:5, :5]


In [ ]:
plt.figure(figsize=(8, 5.5))
image = plt.imshow(np.log10(phase), aspect='auto', origin='lower',
                   extent=[tau_grid.min(), tau_grid.max(), np.log10(eps_grid.min()), np.log10(eps_grid.max())])
plt.colorbar(image, label='log10 first-specialization capacity')
plt.xlabel('Shortcut richness τ')
plt.ylabel('log10 shared-space distortion ε')
plt.title('Specialization threshold phase diagram')
plt.show()


# 8. What this experiment establishes

### Established by the toy allocation model

- The constrained resource-allocation formulation has an exact solution for the power-law case.
- Symmetric languages can produce a global crossover without a gradual cascade.
- Heterogeneous \((\tau_L,\varepsilon_L)\) values produce distinct specialization thresholds and can generate a cascade.
- The model predicts
  \[
  \tau\uparrow\Rightarrow N^*\downarrow,
  \qquad
  \varepsilon\uparrow\Rightarrow N^*\downarrow.
  \]

### Not established

This notebook does **not** show that SGD in a standard Transformer necessarily solves this allocation problem.

That is the next experiment.

# 9. Next experiment

**Experiment 2: a tiny learnable shared/private model.**

Requirements:

1. one shared bottleneck of capacity \(N_S\);
2. optional language-private modules \(N_L\);
3. fixed total parameter budget;
4. an invariant synthetic task;
5. an explicit \(\varepsilon\) manipulation;
6. an explicit \(\tau\) manipulation.

The primary success criterion is not merely that private representations appear. It is:

\[
\boxed{\text{empirical specialization threshold moves in the predicted direction when }\tau\text{ or }\varepsilon\text{ is manipulated.}}
\]

Only after this passes should we return to the PCFG/Transformer setup and then to real multilingual LMs.